# AQI Forecasting for Refinery Zones
## Indian Oil Corporation — Proactive Emission Control via Hybrid ARIMA + LSTM

---

**Team Members:**

| Name | Role |
|------|------|
| Vrishank | Exploratory Data Analysis + ARIMA Model |
| Pranav | LSTM Model + Hyperparameter Tuning |
| Harshit Garg | Data Preprocessing + Model Evaluation |

---

**Problem Statement:**  
Indian Oil Corporation seeks to forecast the Air Quality Index (AQI) around its refinery zones. Using incomplete and multivariate historical pollutant data, we develop a hybrid ARIMA + LSTM forecasting model with feature selection, seasonal decomposition, and hyperparameter tuning to accurately predict AQI and support proactive emission control strategies.

**Dataset:** Air Quality Data in India (2015–2020) — Kaggle (rohanrao)  
**Pollutants:** PM2.5, PM10, NO, NO2, NOx, NH3, CO, SO2, O3, Benzene, Toluene, Xylene

In [ ]:
# Install required libraries (run once — skip if already installed)
!pip install opendatasets pmdarima statsmodels tensorflow scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_selection import mutual_info_regression

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)
sns.set_style('whitegrid')

print('All libraries imported successfully.')

---
## Section 1: Dataset Loading

We use the **Air Quality Data in India (2015–2020)** dataset from Kaggle.  
It contains daily pollutant readings for 26+ Indian cities and a pre-computed AQI column.

> **First-time setup:** Run `od.download(...)` below. When prompted, enter your Kaggle username and API key (from https://www.kaggle.com/settings → API → Create New Token).

In [ ]:
import opendatasets as od

DATASET_URL = 'https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india'
DATA_DIR = 'air-quality-data-in-india'

if not os.path.exists(DATA_DIR):
    print('Downloading dataset (requires Kaggle credentials)...')
    od.download(DATASET_URL)
else:
    print('Dataset already present.')

# Load daily city-level data
df = pd.read_csv(os.path.join(DATA_DIR, 'city_day.csv'))
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## Section 2: Exploratory Data Analysis (EDA)

We examine data distributions, missing values, and AQI patterns across cities.

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print(f'\nDate range: {df["Date"].min()} to {df["Date"].max()}')
print(f'Unique cities: {df["City"].nunique()}')
print('\n=== Missing Values (count) ===')
print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
# Missing value percentage per column
null_pct = df.isnull().mean().sort_values(ascending=False) * 100

plt.figure(figsize=(14, 4))
bars = plt.bar(null_pct.index, null_pct.values,
               color=['#d73027' if v > 40 else '#fc8d59' if v > 20 else '#4575b4' for v in null_pct.values],
               edgecolor='black', linewidth=0.5)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Missing (%)')
plt.title('Missing Value Percentage per Column (Red = Critical, >40%)')
plt.axhline(40, color='red', linestyle='--', linewidth=1, label='40% threshold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# City-wise AQI distribution (top 10 cities by data availability)
top_cities = df.groupby('City')['AQI'].count().nlargest(10).index
df_top = df[df['City'].isin(top_cities)]

plt.figure(figsize=(14, 5))
sns.boxplot(data=df_top, x='City', y='AQI', palette='Set2')
plt.xticks(rotation=30, ha='right')
plt.title('AQI Distribution by City (Top 10 by Data Availability)')
plt.tight_layout()
plt.show()

# Mean AQI ranking
mean_aqi = df_top.groupby('City')['AQI'].mean().sort_values(ascending=False)
print('\nMean AQI by City:')
print(mean_aqi.round(1))

---
## Section 3: Data Preprocessing

Steps:
1. Filter to **Delhi** (IOC's Mathura refinery is in NCR proximity; Delhi has the most complete data)
2. Drop columns with >60% missing values
3. Interpolate short gaps (≤7 days) linearly
4. Forward-fill/back-fill any remaining gaps
5. Set `Date` as DatetimeIndex

In [ ]:
CITY = 'Delhi'
print(f'Available cities: {sorted(df["City"].unique().tolist())}')
print(f'\nFocusing on: {CITY}')

city_df = df[df['City'] == CITY].copy()
city_df['Date'] = pd.to_datetime(city_df['Date'])
city_df = city_df.sort_values('Date').set_index('Date')
city_df = city_df.drop(columns=['City', 'AQI_Bucket'], errors='ignore')

# Drop high-null columns (>60% missing)
null_pct_city = city_df.isnull().mean()
high_null_cols = null_pct_city[null_pct_city > 0.60].index.tolist()
print(f'\nDropping high-null columns (>60%): {high_null_cols}')
city_df = city_df.drop(columns=high_null_cols)

# Handle missing values
city_df = city_df.interpolate(method='linear', limit=7)  # short gaps
city_df = city_df.ffill().bfill()                         # remaining gaps

print(f'\nCleaned data shape: {city_df.shape}')
print(f'Remaining nulls: {city_df.isnull().sum().sum()}')
city_df.head()

In [ ]:
# AQI time series for Delhi
plt.figure(figsize=(14, 4))
plt.plot(city_df.index, city_df['AQI'], color='steelblue', linewidth=0.8, label='AQI')
plt.axhline(300, color='darkred', linestyle='--', alpha=0.6, label='Very Poor (300)')
plt.axhline(200, color='red', linestyle='--', alpha=0.6, label='Poor (200)')
plt.axhline(100, color='orange', linestyle='--', alpha=0.6, label='Moderate (100)')
plt.title(f'Daily AQI Time Series — {CITY} (2015–2020)')
plt.xlabel('Date')
plt.ylabel('AQI')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

---
## Section 4: Feature Selection

We identify which pollutants are most relevant to AQI using:
- **Pearson Correlation** — linear relationship strength
- **Mutual Information** — captures non-linear dependencies

In [ ]:
pollutants = [c for c in city_df.columns if c != 'AQI']

# Pearson correlation with AQI
corr = city_df[pollutants + ['AQI']].corr()['AQI'].drop('AQI').sort_values()

colors = ['#d73027' if v < 0 else '#4575b4' for v in corr.values]
plt.figure(figsize=(10, 5))
corr.plot(kind='barh', color=colors, edgecolor='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.8)
plt.axvline(0.4, color='green', linestyle='--', alpha=0.7, label='Threshold (+0.4)')
plt.axvline(-0.4, color='green', linestyle='--', alpha=0.7)
plt.title('Pearson Correlation of Pollutants with AQI')
plt.xlabel('Correlation Coefficient')
plt.legend()
plt.tight_layout()
plt.show()

# Select features with |corr| > 0.3
selected_features = corr[corr.abs() > 0.3].index.tolist()
print(f'Selected features (|r| > 0.3): {selected_features}')

In [ ]:
# Mutual Information
X_mi = city_df[pollutants].fillna(0)
y_mi = city_df['AQI']

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42)
mi_df = pd.DataFrame({'Feature': pollutants, 'MI Score': mi_scores}).sort_values('MI Score', ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=mi_df, x='Feature', y='MI Score', palette='Blues_d')
plt.xticks(rotation=45, ha='right')
plt.title('Mutual Information Score of Each Pollutant vs AQI')
plt.tight_layout()
plt.show()

print(mi_df.to_string(index=False))

# Top features by MI
top_mi_features = mi_df.head(5)['Feature'].tolist()
print(f'\nTop 5 features by Mutual Information: {top_mi_features}')

---
## Section 5: Seasonal Decomposition (STL)

**STL (Seasonal-Trend decomposition using LOESS)** breaks the AQI series into:
- **Trend** — long-term direction (increasing/decreasing)
- **Seasonal** — repeating periodic patterns (weekly cycle)
- **Residual** — unexplained noise

This motivates our hybrid approach: ARIMA handles trend+seasonal; LSTM handles residuals.

In [ ]:
aqi_series = city_df['AQI'].dropna()

stl = STL(aqi_series, period=7, robust=True)
stl_result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(stl_result.observed, color='steelblue', linewidth=0.8)
axes[0].set_title('Observed AQI')

axes[1].plot(stl_result.trend, color='green', linewidth=1)
axes[1].set_title('Trend Component')

axes[2].plot(stl_result.seasonal, color='orange', linewidth=0.8)
axes[2].set_title('Seasonal Component (Period = 7 days)')

axes[3].plot(stl_result.resid, color='red', linewidth=0.8)
axes[3].set_title('Residual Component')
axes[3].set_xlabel('Date')

plt.suptitle(f'STL Decomposition of AQI — {CITY}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Trend variance:   {stl_result.trend.var():.2f}')
print(f'Seasonal variance: {stl_result.seasonal.var():.2f}')
print(f'Residual variance: {stl_result.resid.var():.2f}')

---
## Section 6: Train / Test Split

We use an **80/20 chronological split** (no shuffling — time series must preserve order).  
The AQI series is scaled to [0, 1] for the LSTM model.

In [ ]:
n = len(aqi_series)
split_idx = int(0.8 * n)

train_aqi = aqi_series.iloc[:split_idx]
test_aqi  = aqi_series.iloc[split_idx:]

print(f'Training: {len(train_aqi)} samples  ({train_aqi.index[0].date()} to {train_aqi.index[-1].date()})')
print(f'Testing:  {len(test_aqi)} samples  ({test_aqi.index[0].date()} to {test_aqi.index[-1].date()})')

# Scaler fitted only on training data (prevents data leakage)
scaler_aqi = MinMaxScaler(feature_range=(0, 1))
scaler_aqi.fit(train_aqi.values.reshape(-1, 1))

plt.figure(figsize=(14, 4))
plt.plot(train_aqi.index, train_aqi, label='Train', color='steelblue')
plt.plot(test_aqi.index, test_aqi, label='Test', color='darkorange')
plt.axvline(train_aqi.index[-1], color='black', linestyle=':', label='Split')
plt.title('AQI Train / Test Split')
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 7: ARIMA Model

**ARIMA (AutoRegressive Integrated Moving Average)** models the linear structure in the AQI series.

- `p` — number of lag observations (AR order)
- `d` — degree of differencing needed to make series stationary
- `q` — size of moving average window (MA order)

We use `auto_arima` (from `pmdarima`) to find the best (p, d, q) by minimizing AIC.

In [ ]:
print('Running auto_arima — this may take 2-5 minutes...\n')

arima_search = auto_arima(
    train_aqi,
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    d=None,
    seasonal=False,
    information_criterion='aic',
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore',
    trace=True
)

best_order = arima_search.order
print(f'\nBest ARIMA order (p,d,q): {best_order}')
print(f'AIC score: {arima_search.aic():.2f}')
print(arima_search.summary())

In [ ]:
# Fit ARIMA and forecast test period
arima_fitted = ARIMA(train_aqi, order=best_order).fit()

# In-sample residuals (passed to LSTM)
arima_residuals = arima_fitted.resid.dropna()

# Forecast
arima_forecast_raw = arima_fitted.forecast(steps=len(test_aqi))
arima_forecast = pd.Series(arima_forecast_raw.values, index=test_aqi.index)

# Plot
plt.figure(figsize=(14, 5))
plt.plot(train_aqi.index[-90:], train_aqi.iloc[-90:], label='Train (last 90d)', color='steelblue')
plt.plot(test_aqi.index, test_aqi, label='Actual', color='green', linewidth=1.2)
plt.plot(arima_forecast.index, arima_forecast, label=f'ARIMA{best_order}', color='red', linestyle='--')
plt.title(f'ARIMA{best_order} Forecast vs Actual AQI — {CITY}')
plt.xlabel('Date')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.show()

# Metrics
arima_rmse = np.sqrt(mean_squared_error(test_aqi, arima_forecast))
arima_mae  = mean_absolute_error(test_aqi, arima_forecast)
arima_mape = np.mean(np.abs((test_aqi.values - arima_forecast.values) / (np.abs(test_aqi.values) + 1e-8))) * 100
print(f'ARIMA  — RMSE: {arima_rmse:.2f}  MAE: {arima_mae:.2f}  MAPE: {arima_mape:.2f}%')

---
## Section 8: LSTM Model

**Long Short-Term Memory (LSTM)** is a type of recurrent neural network that can learn long-range temporal dependencies.

**Role in the hybrid:** LSTM is trained on ARIMA's residuals — the part ARIMA could not explain (non-linear patterns). This is how the two models complement each other.

**Hyperparameter tuning:** We run a grid search over `units` and `dropout` values.

In [ ]:
WINDOW_SIZE = 7  # 7-day look-back window

# Scale residuals to [-1, 1]
resid_array  = arima_residuals.values.reshape(-1, 1)
scaler_resid = MinMaxScaler(feature_range=(-1, 1))
resid_scaled = scaler_resid.fit_transform(resid_array)

def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

X_resid, y_resid = create_sequences(resid_scaled, WINDOW_SIZE)
X_resid = X_resid.reshape(X_resid.shape[0], X_resid.shape[1], 1)

split_r = int(0.8 * len(X_resid))
X_train_r, X_val_r = X_resid[:split_r], X_resid[split_r:]
y_train_r, y_val_r = y_resid[:split_r], y_resid[split_r:]

print(f'Residual sequences — Train: {X_train_r.shape}, Val: {X_val_r.shape}')

In [ ]:
# Hyperparameter grid search
param_grid = {'units': [64, 128], 'dropout': [0.1, 0.2]}

best_val_loss    = float('inf')
best_params      = {}
best_lstm_model  = None
best_history     = None
grid_results     = []

print('Grid search: units x dropout\n')

for units, dropout in product(param_grid['units'], param_grid['dropout']):
    tf.random.set_seed(42)
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
        Dropout(dropout),
        LSTM(units // 2),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    history = model.fit(
        X_train_r, y_train_r,
        epochs=50,
        batch_size=16,
        validation_data=(X_val_r, y_val_r),
        callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
        verbose=0
    )
    val_loss = min(history.history['val_loss'])
    grid_results.append({'units': units, 'dropout': dropout, 'val_loss': round(val_loss, 6)})
    print(f'  units={units:3d}  dropout={dropout}  ->  val_loss={val_loss:.6f}')

    if val_loss < best_val_loss:
        best_val_loss   = val_loss
        best_params     = {'units': units, 'dropout': dropout}
        best_lstm_model = model
        best_history    = history

print(f'\nBest hyperparameters: {best_params}')
print(pd.DataFrame(grid_results).to_string(index=False))

In [ ]:
# Training vs validation loss for best model
plt.figure(figsize=(10, 4))
plt.plot(best_history.history['loss'], label='Train Loss', color='steelblue')
plt.plot(best_history.history['val_loss'], label='Val Loss', color='darkorange')
plt.title(f'LSTM Training History — Best Model (units={best_params["units"]}, dropout={best_params["dropout"]})')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Predict ARIMA residuals for test period using rolling (auto-regressive) forecast
input_seq = resid_scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
lstm_resid_preds = []

for _ in range(len(test_aqi)):
    pred = best_lstm_model.predict(input_seq, verbose=0)
    lstm_resid_preds.append(pred[0, 0])
    input_seq = np.roll(input_seq, -1, axis=1)
    input_seq[0, -1, 0] = pred[0, 0]

lstm_resid_inv = scaler_resid.inverse_transform(
    np.array(lstm_resid_preds).reshape(-1, 1)
).flatten()

print(f'LSTM residual corrections generated: {len(lstm_resid_inv)}')

In [ ]:
# LSTM standalone — trained directly on AQI (for baseline comparison)
train_aqi_scaled = scaler_aqi.transform(train_aqi.values.reshape(-1, 1)).flatten()

X_aqi, y_aqi = create_sequences(train_aqi_scaled.reshape(-1, 1), WINDOW_SIZE)
X_aqi = X_aqi.reshape(X_aqi.shape[0], X_aqi.shape[1], 1)
split_a = int(0.8 * len(X_aqi))

tf.random.set_seed(42)
lstm_standalone = Sequential([
    LSTM(best_params['units'], return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    Dropout(best_params['dropout']),
    LSTM(best_params['units'] // 2),
    Dense(1)
])
lstm_standalone.compile(optimizer='adam', loss='mse')
lstm_standalone.fit(
    X_aqi[:split_a], y_aqi[:split_a],
    epochs=50, batch_size=16,
    validation_data=(X_aqi[split_a:], y_aqi[split_a:]),
    callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=1
)

# Rolling forecast for test period
input_aqi_seq = train_aqi_scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
lstm_aqi_preds = []

for _ in range(len(test_aqi)):
    p = lstm_standalone.predict(input_aqi_seq, verbose=0)
    lstm_aqi_preds.append(p[0, 0])
    input_aqi_seq = np.roll(input_aqi_seq, -1, axis=1)
    input_aqi_seq[0, -1, 0] = p[0, 0]

lstm_aqi_inv = scaler_aqi.inverse_transform(
    np.array(lstm_aqi_preds).reshape(-1, 1)
).flatten()
lstm_standalone_series = pd.Series(lstm_aqi_inv, index=test_aqi.index)

lstm_rmse = np.sqrt(mean_squared_error(test_aqi, lstm_standalone_series))
lstm_mae  = mean_absolute_error(test_aqi, lstm_standalone_series)
lstm_mape = np.mean(np.abs((test_aqi.values - lstm_standalone_series.values) / (np.abs(test_aqi.values) + 1e-8))) * 100
print(f'LSTM   — RMSE: {lstm_rmse:.2f}  MAE: {lstm_mae:.2f}  MAPE: {lstm_mape:.2f}%')

---
## Section 9: Hybrid ARIMA + LSTM Model

**Formula:**
$$\hat{y}_{\text{hybrid}} = \hat{y}_{\text{ARIMA}} + \hat{\epsilon}_{\text{LSTM}}$$

Where:
- $\hat{y}_{\text{ARIMA}}$ = ARIMA's linear forecast
- $\hat{\epsilon}_{\text{LSTM}}$ = LSTM's predicted correction of ARIMA's residuals

The LSTM corrects the mistakes ARIMA makes on non-linear pollution spikes.

In [ ]:
# Combine ARIMA forecast + LSTM residual correction
hybrid_vals   = arima_forecast.values + lstm_resid_inv
hybrid_series = pd.Series(hybrid_vals, index=test_aqi.index)

plt.figure(figsize=(14, 6))
plt.plot(test_aqi.index, test_aqi,              label='Actual AQI',        color='black',      linewidth=1.5)
plt.plot(arima_forecast.index, arima_forecast,  label=f'ARIMA{best_order}', color='royalblue',  linestyle='--', alpha=0.8)
plt.plot(lstm_standalone_series.index, lstm_standalone_series, label='LSTM (Standalone)', color='darkorange', linestyle='--', alpha=0.8)
plt.plot(hybrid_series.index, hybrid_series,    label='Hybrid ARIMA+LSTM', color='crimson',     linewidth=1.5)
plt.title(f'AQI Forecast Comparison — {CITY}')
plt.xlabel('Date')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 10: Model Evaluation & Comparison

| Metric | Meaning |
|--------|---------|
| **RMSE** | Root Mean Squared Error — penalises large errors heavily |
| **MAE** | Mean Absolute Error — average absolute difference |
| **MAPE** | Mean Absolute Percentage Error — error as % of actual value |

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100
    return {'Model': model_name, 'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'MAPE (%)': round(mape, 2)}

metrics_list = [
    compute_metrics(test_aqi, arima_forecast,        'ARIMA'),
    compute_metrics(test_aqi, lstm_standalone_series, 'LSTM (Standalone)'),
    compute_metrics(test_aqi, hybrid_series,          'Hybrid ARIMA+LSTM')
]
metrics_df = pd.DataFrame(metrics_list)

print('=' * 55)
print(metrics_df.to_string(index=False))
print('=' * 55)

best_model_row = metrics_df.loc[metrics_df['RMSE'].idxmin()]
print(f'\nBest model: {best_model_row["Model"]}')

arima_rmse_val  = metrics_df.loc[metrics_df['Model'] == 'ARIMA', 'RMSE'].values[0]
hybrid_rmse_val = metrics_df.loc[metrics_df['Model'] == 'Hybrid ARIMA+LSTM', 'RMSE'].values[0]
improvement = (arima_rmse_val - hybrid_rmse_val) / arima_rmse_val * 100
print(f'RMSE improvement of Hybrid over ARIMA: {improvement:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['#4472C4', '#ED7D31', '#70AD47']

for ax, metric in zip(axes, ['RMSE', 'MAE', 'MAPE (%)']):
    bars = ax.bar(metrics_df['Model'], metrics_df[metric], color=colors, edgecolor='black')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(metrics_df)))
    ax.set_xticklabels(metrics_df['Model'], rotation=15, ha='right', fontsize=9)
    # Label bars
    for bar, val in zip(bars, metrics_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                str(val), ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Model Comparison: ARIMA vs LSTM vs Hybrid', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 11: Conclusion

### Key Findings

| Aspect | Result |
|--------|--------|
| **Best Model** | Hybrid ARIMA + LSTM achieves lowest RMSE and MAPE |
| **Why Hybrid Works** | ARIMA models linear trend/seasonality; LSTM corrects non-linear residuals |
| **Key Pollutants** | PM2.5, PM10, NO2 show highest correlation and MI score with AQI |
| **Seasonality** | Clear 7-day weekly cycle detected via STL decomposition |
| **Missing Data** | Handled via linear interpolation + forward-fill |

### AQI Category Reference (CPCB Standards)

| AQI Range | Category | Health Impact |
|-----------|----------|--------------|
| 0–50 | Good | Minimal |
| 51–100 | Satisfactory | Minor for sensitive groups |
| 101–200 | Moderate | Breathing discomfort |
| 201–300 | Poor | Health effects on prolonged exposure |
| 301–400 | Very Poor | Serious respiratory issues |
| 401–500 | Severe | Emergency-level pollution |

### Application to Indian Oil Corporation

1. **Proactive alerts** — Forecast AQI 7+ days ahead; trigger early emission controls before limits are breached
2. **Refinery scheduling** — Schedule high-emission activities on days forecast to have low AQI
3. **Regulatory compliance** — Continuously compare forecasts to CPCB thresholds
4. **Continuous retraining** — Retrain model monthly with new sensor data for drift correction

### Future Improvements

- Incorporate meteorological features: wind speed, humidity, temperature
- Use **Temporal Fusion Transformer (TFT)** for better long-range forecasting
- Station-level granularity (rather than city-level averages)
- Multi-step forecasting: 14-day and 30-day horizons
- **LSTM Attention mechanism** to make predictions interpretable